# GAN：生成器与判别器的博弈
> 难度标签：高级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：06_深度学习 | 源文件：生成对抗网络_GAN.py | 核心功能：对抗训练、生成器/判别器、模式崩溃

## 概述

GAN 由生成器 G 和判别器 D 组成。G 试图生成以假乱真的样本，D 试图区分真假。两者博弈训练，最终 G 能生成逼真数据。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
# --- 导入库 ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

## 数学原理

### 1. GAN 的博弈论框架

GAN 由生成器 $G$ 和判别器 $D$ 组成，形成一个**极小极大博弈**：

$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

- $D(x)$：判别器判断 $x$ 为真实数据的概率
- $G(z)$：生成器从噪声 $z \sim p_z(z)$ 生成的假样本
- $D$ 想最大化 $V$（正确区分真假），$G$ 想最小化 $V$（骗过 $D$）

### 2. 判别器的最优解

固定 $G$ 时，最优判别器为：

$$D^*(x) = \frac{p_{data}(x)}{p_{data}(x) + p_g(x)}$$

其中 $p_g$ 是生成器隐式定义的数据分布。直觉：当 $p_{data}(x) = p_g(x)$ 时，$D^*(x) = 0.5$，即判别器完全无法区分。

### 3. 全局最优与 JS 散度

将 $D^*$ 代入目标函数，得到：

$$C(G) = -\log 4 + 2 \cdot JSD(p_{data} \| p_g)$$

其中 $JSD$ 是 **Jensen-Shannon 散度**：

$$JSD(P \| Q) = \frac{1}{2} KL(P \| M) + \frac{1}{2} KL(Q \| M), \quad M = \frac{P + Q}{2}$$

当且仅当 $p_g = p_{data}$ 时，$JSD = 0$，$C(G)$ 取最小值 $-\log 4$。

### 4. 判别器损失

实际训练中，判别器的损失函数为：

$$\mathcal{L}_D = -\frac{1}{m}\sum_{i=1}^{m}\left[\log D(x^{(i)}) + \log(1 - D(G(z^{(i)})))\right]$$

对应的代码中 `criterion = nn.BCELoss()` 实现了二元交叉熵。

### 5. 生成器损失

原始论文的生成器损失：

$$\mathcal{L}_G = \frac{1}{m}\sum_{i=1}^{m}\log(1 - D(G(z^{(i)})))$$

实践中常使用**非饱和启发**（non-saturating heuristic）以避免梯度消失：

$$\mathcal{L}_G^{ns} = -\frac{1}{m}\sum_{i=1}^{m}\log D(G(z^{(i)}))$$

即让生成器最大化"判别器把假样本判为真"的概率。

### 6. 训练动态分析

**训练平衡**：$D$ 和 $G$ 需要交替训练，保持能力平衡。

| 问题 | 表现 | 数学原因 |
|------|------|----------|
| $D$ 太强 | $D(G(z)) \approx 0$，$G$ 梯度消失 | $\log(1-D(G(z)))$ 饱和 |
| $G$ 太强 | 模式崩塌（mode collapse） | $G$ 只学会生成少数样本 |
| 理想平衡 | $D(x) \approx 0.5$ | $p_g \approx p_{data}$ |

代码中 `betas=(0.5, 0.999)` 是 GAN 特有的 Adam 参数，一阶动量降低以稳定训练。

### 7. 重参数化与采样

生成器的输入噪声通常采样自：

$$z \sim \mathcal{N}(0, I) \quad \text{或} \quad z \sim \text{Uniform}(-1, 1)$$

代码中 `latent_dim=32` 对应噪声向量维度。生成器学习的是一个从低维噪声空间到高维数据空间的映射 $G: \mathcal{Z} \to \mathcal{X}$。

### 8. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `Generator(latent_dim=32)` | $G(z): \mathbb{R}^{32} \to \mathbb{R}^{64}$ |
| `Discriminator(input_dim=64)` | $D(x): \mathbb{R}^{64} \to [0,1]$ |
| `BCELoss` 用于 $D$ | $\mathcal{L}_D = -[\log D(x) + \log(1-D(G(z)))]$ |
| `BCELoss` 用于 $G$（标签=1） | $\mathcal{L}_G^{ns} = -\log D(G(z))$ |
| `Tanh()` 输出层 | 将生成样本归一化到 $[-1, 1]$ |
| `LeakyReLU(0.2)` | 避免判别器梯度稀疏，$\text{LReLU}(x) = \max(0.2x, x)$ |
| `betas=(0.5, 0.999)` | 降低 Adam 一阶动量以稳定 GAN 训练 |

### 1. 加载数据

首先加载数据集，为后续训练和评估做准备。

In [ ]:
digits = load_digits()
X = StandardScaler().fit_transform(digits.data)
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
X_train_t = torch.FloatTensor(X_train)
train_loader = DataLoader(TensorDataset(X_train_t), batch_size=64, shuffle=True)

### 2. 生成器

运行 2. 生成器 的代码，观察算法在该环节的行为。

In [ ]:
class Generator(nn.Module):
    """随机噪声 z → 生成假样本"""
    def __init__(self, latent_dim=32, output_dim=64):
        super().__init__()
        self.net = nn.Sequential(
# --- 继续 ---
            nn.Linear(latent_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, output_dim),
# --- 继续 ---
            nn.Tanh(),  # 输出 [-1, 1]
        )

    def forward(self, z):
        return self.net(z)

### 3. 判别器

运行 3. 判别器 的代码，观察算法在该环节的行为。

In [ ]:
class Discriminator(nn.Module):
    """输入样本 → 输出真/假概率"""
    def __init__(self, input_dim=64):
        super().__init__()
        self.net = nn.Sequential(
# --- 继续 ---
            nn.Linear(input_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
# --- 继续 ---
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

### 4. 初始化

运行 4. 初始化 的代码，观察算法在该环节的行为。

In [ ]:
latent_dim = 32
G = Generator(latent_dim=latent_dim)
D = Discriminator(input_dim=64)

criterion = nn.BCELoss()
opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

print("=== GAN 模型 ===")
print(f"生成器参数: {sum(p.numel() for p in G.parameters()):,}")
print(f"判别器参数: {sum(p.numel() for p in D.parameters()):,}")

### 5. 训练

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
print("\n=== 训练 ===")
n_epochs = 100
d_losses, g_losses = [], []

for epoch in range(n_epochs):
    epoch_d_loss = 0
    epoch_g_loss = 0

    for (real_data,) in train_loader:
        batch_size = real_data.size(0)
        real_labels = torch.ones(batch_size, 1)
        fake_labels = torch.zeros(batch_size, 1)

        # === 训练判别器 ===
        z = torch.randn(batch_size, latent_dim)
        fake_data = G(z).detach()  # 切断生成器梯度

        d_real = D(real_data)
        d_fake = D(fake_data)
        d_loss = criterion(d_real, real_labels) + criterion(d_fake, fake_labels)

        opt_D.zero_grad()
        d_loss.backward()
        opt_D.step()

        # === 训练生成器 ===
        z = torch.randn(batch_size, latent_dim)
        fake_data = G(z)
        d_fake = D(fake_data)
        g_loss = criterion(d_fake, real_labels)  # 生成器想让判别器判为真

        opt_G.zero_grad()
        g_loss.backward()
        opt_G.step()

        epoch_d_loss += d_loss.item()
        epoch_g_loss += g_loss.item()

    d_losses.append(epoch_d_loss / len(train_loader))
    g_losses.append(epoch_g_loss / len(train_loader))

    if (epoch + 1) % 20 == 0:
        # 计算判别器准确率
        with torch.no_grad():
            z = torch.randn(len(X_test), latent_dim)
            fake = G(z)
            d_real_acc = (D(torch.FloatTensor(X_test)) > 0.5).float().mean().item()
            d_fake_acc = (D(fake) < 0.5).float().mean().item()
# --- 输出结果 ---
        print(f"  Epoch {epoch+1:>3}: D_loss={d_losses[-1]:.4f}, G_loss={g_losses[-1]:.4f}, "
              f"D_acc: real={d_real_acc:.2f}, fake={d_fake_acc:.2f}")

### 6. 生成新样本

运行 6. 生成新样本 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 生成新样本 ===")
G.eval()
with torch.no_grad():
    z = torch.randn(10, latent_dim)
    generated = G(z).numpy()
# --- 输出结果 ---
    print(f"生成 {generated.shape[0]} 个样本，像素范围: [{generated.min():.3f}, {generated.max():.3f}]")

### 7. 训练动态

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
print("\n=== 训练动态 ===")
print(f"最终 D_loss: {d_losses[-1]:.4f}, G_loss: {g_losses[-1]:.4f}")
print("理想状态: D_loss ≈ ln(4) ≈ 1.386, D(real) ≈ 0.5, D(fake) ≈ 0.5")

print("\n=== GAN 要点 ===")
print("生成器 G: 噪声 z → 假样本，目标：骗过判别器")
print("判别器 D: 样本 → 真/假，目标：区分真假")
print("博弈目标: min_G max_D V(D,G) = E[log D(x)] + E[log(1-D(G(z)))]")
print()
# --- 输出结果 ---
print("常见问题:")
print("1. 模式坍塌: 生成器只生成少数样本")
print("2. 训练不稳定: D 太强 → G 梯度消失；D 太弱 → G 无指导")
print("3. 改进: WGAN, Progressive GAN, StyleGAN")

## 关键代码解释

### 交替训练

```python
# 判别器：区分真假
d_loss = -torch.mean(torch.log(D(real)) + torch.log(1 - D(G(noise))))
# 生成器：骗过判别器
g_loss = -torch.mean(torch.log(D(G(noise))))
```

### 训练技巧

```python
# 实际中用 BCE loss 替代原始公式
# 标签平滑：real_label = 0.9
# 噪声输入：给判别器输入加噪声
```

## 注意事项

1. **训练不稳定**：GAN 训练是出了名的难
2. **模式崩溃**：生成器只生成少数几种样本
3. **评估困难**：没有像 loss 这样直观的评估指标

## 总结与延伸

以上代码展示了 **生成对抗网络 GAN** 的完整流程。

**解读要点**：
- 关注 **损失函数** 的收敛曲线
- 观察训练/验证损失是否出现分叉（过拟合信号）
- 对比不同超参数（学习率、层数）的影响

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **WGAN**：用 Wasserstein 距离替代 JS 散度，训练更稳定
- **StyleGAN**：高质量图像生成
- **扩散模型**：正在取代 GAN 成为主流生成模型
